# Overfit MeshGraphNet on a couple of bumper sims (Colab)

Sanity check for the VTKHDF dataloader + crash pipeline: train **MeshGraphNet** (one-shot) on **2 simulations** for many epochs and watch the per-epoch `avg_loss` fall toward ~0.

**What this proves / doesn't:** loss → ~0 means the *pipeline trains end-to-end* (data → PyG graph → model → loss → step). It does **not** prove MeshGraphNet learned the physics — global scalars (velocity / thickness / pole) are broadcast to every node and the structural-only mesh is 16 disconnected components, so 2 samples are easy to memorize.

Use a **GPU runtime** (Runtime → Change runtime type → GPU). The real PhysicsNeMo MeshGraphNet needs `torch>=2.10`, which is why this runs on Colab rather than an Intel Mac.

## 1. Install: clone the fork + dependencies

If `pip install -e .` tries to downgrade/upgrade Colab's torch and breaks, instead `pip install nvidia-physicsnemo` and rely on the cloned example code on `sys.path`.

In [ ]:
!git clone --depth 1 https://github.com/tum-ai/physicsnemo.git /content/physicsnemo
%cd /content/physicsnemo
!pip install -q -e .
!pip install -q torch_geometric
!pip install -q -r examples/structural_mechanics/crash/requirements.txt
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 2. Get the data

The full dataset (~1 GB) is **not** in the repo. You only need **2–5 sim folders** plus `bumper_beam_master_with_split.csv`. Easiest: upload them to Google Drive, e.g.

```
MyDrive/bumper/
  bumper_beam_master_with_split.csv
  simulations/
    sim_00001/sim_00001.vtkhdf
    sim_00002/sim_00002.vtkhdf
```

Then mount Drive and point `DATA_DIR` / `MASTER_CSV` at it. (Or use the file-upload widget for a couple of files.)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR   = '/content/drive/MyDrive/bumper/simulations'
MASTER_CSV = '/content/drive/MyDrive/bumper/bumper_beam_master_with_split.csv'
assert os.path.isdir(DATA_DIR) and os.path.isfile(MASTER_CSV), 'fix DATA_DIR / MASTER_CSV'
print('sims found:', sorted(os.listdir(DATA_DIR))[:5])

## 3. Build the global-features JSON

Maps the master-CSV columns to the 3-key contract (`velocity_x`, `thickness_scale`, `rwall_origin_y`).

In [ ]:
%cd /content/physicsnemo/examples/structural_mechanics/crash
!python make_global_features.py --master-csv "$MASTER_CSV" --out ./global_features.json

## 4. Overfit MeshGraphNet on 2 sims

Uses `conf/bumper_vtkhdf_mgn_oneshot.yaml` (graph datapipe + `mgn_one_shot`, `output_dim=500`). Paths are overridden on the CLI so the config's `./simulations` default isn't required.

In [ ]:
import os
os.environ['DATA_DIR'] = DATA_DIR
os.environ['MASTER_CSV'] = MASTER_CSV
# reader auto-discovers the master CSV next to the data dir; copy it there if needed:
!cp "$MASTER_CSV" "$(dirname "$DATA_DIR")/bumper_beam_master_with_split.csv" 2>/dev/null || true

!python train.py --config-name=bumper_vtkhdf_mgn_oneshot \
    training.raw_data_dir="$DATA_DIR" \
    training.raw_data_dir_validation="$DATA_DIR" \
    training.global_features_filepath=./global_features.json \
    training.num_training_samples=2 \
    training.num_validation_samples=0 \
    training.epochs=2000

## 5. Watch the loss drop

TensorBoard logs the per-epoch `loss` scalar; a successful overfit drops it orders of magnitude below epoch 1.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/physicsnemo/examples/structural_mechanics/crash/outputs

In [ ]:
# Fallback: parse avg_loss from the run log and plot it.
import glob, re
import matplotlib.pyplot as plt
logs = sorted(glob.glob('/content/physicsnemo/examples/structural_mechanics/crash/outputs/**/*.log', recursive=True))
losses = []
if logs:
    for line in open(logs[-1]):
        m = re.search(r'avg_loss:\s*([0-9.eE+-]+)', line)
        if m:
            losses.append(float(m.group(1)))
if losses:
    plt.figure(figsize=(7,4)); plt.semilogy(losses)
    plt.xlabel('epoch'); plt.ylabel('avg_loss (log)'); plt.title('MeshGraphNet overfit (2 sims)'); plt.grid(True)
    plt.show()
    print(f'epoch 1: {losses[0]:.3e}   last: {losses[-1]:.3e}   ratio: {losses[0]/max(losses[-1],1e-12):.1f}x')
else:
    print('no avg_loss lines found yet - check the training cell output / log path')